# Reflexion [Step 08.02 - Failure as accumulating memory]

> **MLCourse - Agentic AI - LangGraph**

Reflexion (Shinn et al., 2023) adds one idea to an agent loop:

> When an attempt fails, **write down in words why it failed**, and put that note
> in front of the next attempt.

The mechanism is three roles plus a memory:

```
        +---------------------------------------------------+
        v                                                   |
  ACTOR  ->  EVALUATOR  --pass-->  done                     |
                    |                                       |
                    +--fail-->  SELF-REFLECTION --> memory --+
```

- **Actor** attempts the task (a normal tool-using agent).
- **Evaluator** judges the attempt, ideally against something objective.
- **Self-reflection** turns "you were wrong" into "you were wrong *because* X, so
  next time do Y" - a reusable, natural-language lesson.
- **Memory** is a growing list of those lessons, injected into every retry.

### What you'll learn

- Why the reflection must be **carried into the retry**, or it does nothing at all.
- How to build the actor / evaluator / reflector loop in LangGraph.
- Why the memory is an **appending list**, not an overwritten string.
- The failure modes: reflection loops, flattering evaluators, vague lessons, cost.

### Key takeaways

- The evaluator is the weakest link. An LLM grading its own work is barely better
  than no evaluator at all - ground it in execution or a deterministic check.
- Reflections **accumulate**: attempt 3 sees the lessons from attempts 1 and 2.
- Reflexion trades **tokens for accuracy**. It is the most expensive pattern here.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### The shared task world


In [ ]:
# Every notebook in this module attacks THE SAME task with a different reasoning
# pattern, so the comparison in notebook 05 is apples-to-apples.

from langchain_core.tools import tool

# A tiny deterministic "database". Deterministic matters: we need to check
# correctness automatically, without a human reading the answer.
POPULATION = {"tokyo": 13_960_000, "lagos": 15_400_000, "lima": 9_750_000}
AREA_KM2 = {"tokyo": 2194, "lagos": 1171, "lima": 2672}

TOOL_CALLS = {"count": 0}          # instrumentation: how many tool calls happened


@tool
def population(city: str) -> str:
    """Return the population of a city as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(POPULATION.get(city.strip().lower(), "unknown city"))


@tool
def area_km2(city: str) -> str:
    """Return the land area of a city in square kilometres as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(AREA_KM2.get(city.strip().lower(), "unknown city"))


TOOLS = [population, area_km2]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

TASK = (
    "Among Tokyo, Lagos and Lima, which city has the highest population density "
    "(people per square kilometre)? Answer with the city name and the density "
    "rounded to the nearest whole number."
)

# Ground truth, computed here so the notebook can grade itself.
DENSITIES = {c: POPULATION[c] / AREA_KM2[c] for c in POPULATION}
GT_CITY = max(DENSITIES, key=DENSITIES.get)
GT_DENSITY = round(DENSITIES[GT_CITY])

print("Task:", TASK)
print()
for c in sorted(DENSITIES, key=DENSITIES.get, reverse=True):
    print("  %-6s %9d / %5d = %7.0f people/km2" % (c, POPULATION[c], AREA_KM2[c], DENSITIES[c]))
print()
print("Ground truth -> %s, %d" % (GT_CITY.title(), GT_DENSITY))


def grade(answer: str) -> bool:
    """Automatic grader: the answer must name the right city AND the right density.

    The density is accepted within +/-2 to tolerate rounding differences.
    """
    import re
    if not answer:
        return False
    low = answer.lower()
    if GT_CITY not in low:
        return False
    cleaned = low.replace(",", "").replace(".", " ")
    numbers = [int(n) for n in re.findall(r"\d+", cleaned)]
    return any(abs(n - GT_DENSITY) <= 2 for n in numbers)


### Instrumentation: a token and latency meter


In [ ]:
import time


class Meter:
    """Accumulates token usage, call counts and wall-clock time for one run.

    Every pattern in this module is wrapped in one of these, so notebook 05 can
    compare them on identical instrumentation.
    """

    def __init__(self, name):
        self.name = name
        self.input_tokens = 0
        self.output_tokens = 0
        self.llm_calls = 0
        self.tool_calls = 0
        self.seconds = 0.0
        self._t0 = None

    def start(self):
        TOOL_CALLS["count"] = 0
        self._t0 = time.time()
        return self

    def stop(self):
        self.seconds = time.time() - self._t0
        self.tool_calls = TOOL_CALLS["count"]
        return self

    def record(self, message):
        """Add one AIMessage's usage to the totals, then return the message."""
        usage = getattr(message, "usage_metadata", None) or {}
        if usage:
            self.input_tokens += usage.get("input_tokens", 0)
            self.output_tokens += usage.get("output_tokens", 0)
            self.llm_calls += 1
        return message

    def record_all(self, messages):
        """Add usage from every AIMessage in a list (for create_agent results)."""
        for m in messages:
            if getattr(m, "usage_metadata", None):
                self.record(m)
        return messages

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens

    def report(self, answer=None, correct=None):
        print()
        print("=" * 62)
        print("PATTERN : %s" % self.name)
        print("-" * 62)
        print("LLM calls    : %d" % self.llm_calls)
        print("tool calls   : %d" % self.tool_calls)
        print("input tokens : %d" % self.input_tokens)
        print("output tokens: %d" % self.output_tokens)
        print("TOTAL tokens : %d" % self.total_tokens)
        print("latency      : %.1fs" % self.seconds)
        if correct is not None:
            print("correct      : %s" % ("YES" if correct else "NO"))
        print("=" * 62)
        if answer:
            print(answer)
        return self


### 1. A task that actually fails

To show reflection working we need a task the actor gets wrong on attempt 1 more
often than not. Unit conversion is a reliable source of first-attempt errors: the
tools return people and square kilometres, but we ask for people per **hectare**.

In [4]:
REFLEX_TASK = (
    "Using ONLY the tools, compute the population density of Lagos in people per "
    "HECTARE (1 square kilometre = 100 hectares). "
    "Reply with ONLY the number rounded to one decimal place, nothing else."
)

TRUE_ANSWER = round(POPULATION["lagos"] / (AREA_KM2["lagos"] * 100), 1)
print("ground truth:", TRUE_ANSWER, "people/hectare")

ground truth: 131.5 people/hectare


### 2. The state

The critical field is `reflections`, annotated with `operator.add` so each pass
**appends** rather than replaces.

> **Pitfall:** if you make this a plain `str`, every reflection overwrites the last
> one and you have built an expensive retry loop with no memory. The reducer *is*
> the memory.

In [5]:
from typing import Annotated, TypedDict
import operator, re
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent


class ReflexionState(TypedDict):
    task: str
    answer: str                                     # the actor's latest attempt
    verdict: str                                    # "pass" or "fail"
    evidence: str                                   # why the evaluator said that
    reflections: Annotated[list, operator.add]      # <-- THE MEMORY. Appends.
    attempt: int


MAX_ATTEMPTS = 3          # hard cap: reflection loops are the classic failure mode

### 3. The actor

An ordinary tool-using agent - except that it is handed the accumulated reflections
as part of its instructions.

Note the reflections are formatted as **explicit imperative lessons**, not as a raw
transcript of past failures. Raw transcripts confuse models; instructions steer them.

In [6]:
actor_agent = create_agent(
    model=make_llm(max_tokens=400),
    tools=TOOLS,
    system_prompt="You are a careful analyst. Use tools for facts, do the arithmetic yourself.",
)


def actor(state: ReflexionState) -> dict:
    """ACTOR: attempt the task, with every prior lesson in front of it."""
    n = state.get("attempt", 0) + 1
    prompt = state["task"]

    if state["reflections"]:
        lessons = "\n".join("- " + r for r in state["reflections"])
        prompt = ("%s\n\nYou have attempted this before and failed. "
                  "Lessons learned from those attempts:\n%s\n\n"
                  "Apply every lesson above. Do not repeat those mistakes."
                  % (state["task"], lessons))

    print("[actor    ] attempt %d (%d lesson(s) in memory)" % (n, len(state["reflections"])))
    out = actor_agent.invoke({"messages": [("user", prompt)]})
    meter.record_all(out["messages"])
    answer = out["messages"][-1].content.strip()
    print("[actor    ] -> %s" % answer.replace("\n", " ")[:90])
    return {"answer": answer, "attempt": n}

### 4. The evaluator

**This is the part people get wrong.** If you ask an LLM "was your answer good?",
it says yes. Reflexion only works when the evaluator has grounding the actor cannot
argue with.

Sources of grounding, best first:

1. **Executing the output** - run the code, run the tests, hit the API.
2. **A deterministic checker** - a regex, a schema, a known ground truth.
3. **An LLM judge with an explicit rubric and a requirement to quote evidence** (weakest).

Ours parses the number out of the answer and compares it to a value computed from
the same data the tools serve. Grounded, not self-graded. It also **diagnoses** the
error - vague evidence produces vague reflections.

In [7]:
def evaluate(state: ReflexionState) -> dict:
    """EVALUATOR: objective check. No LLM opinion involved."""
    truth = POPULATION["lagos"] / (AREA_KM2["lagos"] * 100)

    nums = re.findall(r"-?\d+(?:\.\d+)?", state["answer"].replace(",", ""))
    if not nums:
        return {"verdict": "fail", "evidence": "no number found in the answer at all"}

    got = float(nums[-1])                       # last number = the stated result
    if abs(got - truth) <= 0.5:
        return {"verdict": "pass",
                "evidence": "answer %.1f matches the expected %.1f" % (got, truth)}

    ratio = got / truth if truth else 0
    if abs(ratio - 100) < 25:
        why = ("the value is about 100x too large, which is exactly the error you get "
               "by reporting people per SQUARE KILOMETRE instead of per hectare")
    elif abs(ratio - 0.01) < 0.005:
        why = "the value is about 100x too small - the conversion was applied backwards"
    else:
        why = "the value does not match under any simple unit conversion"

    return {"verdict": "fail",
            "evidence": "answer %.4g but expected %.1f; %s" % (got, truth, why)}

### 5. The self-reflection node

Takes the evaluator's evidence and turns it into an **imperative lesson for next
time**. The prompt matters here: ask for an instruction, not an apology. "I should
have been more careful" teaches nothing; "Convert km2 to hectares by multiplying by
100 before dividing" teaches everything.

In [8]:
reflect_llm = make_llm(max_tokens=140)


def reflect(state: ReflexionState) -> dict:
    """SELF-REFLECTION: convert a diagnosis into a reusable instruction."""
    prompt = (
        "An assistant attempted this task:\n%s\n\n"
        "Its answer was: %s\n"
        "An objective checker reported: %s\n\n"
        "Write ONE short imperative lesson (max 30 words) that would prevent this "
        "exact mistake on the next attempt. Start with a verb. No apologies, no preamble."
        % (state["task"], state["answer"], state["evidence"])
    )
    lesson = meter.record(safe_invoke(reflect_llm, prompt)).content.strip()
    lesson = lesson.strip('"').replace("\n", " ")
    print("[reflect  ] lesson: %s" % lesson[:110])
    return {"reflections": [lesson]}          # a LIST -> appended by the reducer

### 6. Wiring the loop


In [9]:
def should_retry(state: ReflexionState) -> str:
    if state["verdict"] == "pass":
        print("[route    ] PASS after %d attempt(s)" % state["attempt"])
        return "done"
    if state["attempt"] >= MAX_ATTEMPTS:
        print("[route    ] budget exhausted after %d attempts" % state["attempt"])
        return "done"
    print("[route    ] fail -> reflect and retry")
    return "reflect"


rg = StateGraph(ReflexionState)
rg.add_node("actor", actor)
rg.add_node("evaluate", evaluate)
rg.add_node("reflect", reflect)
rg.add_edge(START, "actor")
rg.add_edge("actor", "evaluate")
rg.add_conditional_edges("evaluate", should_retry, {"reflect": "reflect", "done": END})
rg.add_edge("reflect", "actor")                # <-- the loop back, carrying memory

reflexion_app = rg.compile()
print(reflexion_app.get_graph().draw_ascii())

          +-----------+           
          | __start__ |           
          +-----------+           
                *                 
                *                 
                *                 
            +-------+             
            | actor |             
            +-------+             
           **        **           
         **            *          
        *               **        
+----------+              *       
| evaluate |              *       
+----------+...           *       
      .        ...        *       
      .           ....    *       
      .               ..  *       
+---------+          +---------+  
| __end__ |          | reflect |  
+---------+          +---------+  


### 7. Run it


In [10]:
meter = Meter("Reflexion").start()
out = reflexion_app.invoke({
    "task": REFLEX_TASK, "answer": "", "verdict": "", "evidence": "",
    "reflections": [], "attempt": 0,
})
meter.stop()

correct = out["verdict"] == "pass"
meter.report("final answer: %s" % out["answer"].replace("\n", " ")[:200], correct)

[actor    ] attempt 1 (0 lesson(s) in memory)


[actor    ] -> 1314.9
[route    ] fail -> reflect and retry


[reflect  ] lesson: Verify unit conversions by checking orders of magnitude; dividing by 100 reduces the value by two decimal plac
[actor    ] attempt 2 (1 lesson(s) in memory)


[actor    ] -> 13149.4
[route    ] fail -> reflect and retry


[reflect  ] lesson: Divide the population density per square kilometre by 100 to convert to people per hectare.
[actor    ] attempt 3 (2 lesson(s) in memory)


[actor    ] -> 131.5
[route    ] PASS after 3 attempt(s)

PATTERN : Reflexion
--------------------------------------------------------------
LLM calls    : 11
tool calls   : 9
input tokens : 5535
output tokens: 379
TOTAL tokens : 5914
latency      : 11.5s
correct      : YES
final answer: 131.5


In [11]:
print("attempts used :", out["attempt"])
print("verdict       :", out["verdict"])
print("evidence      :", out["evidence"])
print()
print("ACCUMULATED MEMORY (%d lesson(s)):" % len(out["reflections"]))
for i, r in enumerate(out["reflections"], 1):
    print("  %d. %s" % (i, r))

attempts used : 3
verdict       : pass
evidence      : answer 131.5 matches the expected 131.5

ACCUMULATED MEMORY (2 lesson(s)):
  1. Verify unit conversions by checking orders of magnitude; dividing by 100 reduces the value by two decimal places, not one.
  2. Divide the population density per square kilometre by 100 to convert to people per hectare.


### 8. Proving the reflection is what helped

A skeptic could say a later attempt succeeded by luck. Test it directly: run the
actor on the same task **with** and **without** the accumulated lesson.

(If the first attempt happened to pass, the memory is empty and both runs are the
same - that itself is worth seeing, because it shows Reflexion costs nothing extra
when the actor gets it right first time.)

In [12]:
lesson_block = "\n".join("- " + r for r in out["reflections"]) or "(memory is empty)"
print("lesson(s) under test:")
print(lesson_block)
print()

truth = POPULATION["lagos"] / (AREA_KM2["lagos"] * 100)


def close(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return bool(nums) and abs(float(nums[-1]) - truth) <= 0.5


blind = actor_agent.invoke({"messages": [("user", REFLEX_TASK)]})
meter.record_all(blind["messages"])
blind_ans = blind["messages"][-1].content.strip().replace("\n", " ")
time.sleep(3)

informed = actor_agent.invoke({"messages": [(
    "user",
    "%s\n\nLessons from previous failed attempts:\n%s\nApply every lesson above."
    % (REFLEX_TASK, lesson_block))]})
meter.record_all(informed["messages"])
informed_ans = informed["messages"][-1].content.strip().replace("\n", " ")

print("WITHOUT memory : %-38s correct=%s" % (blind_ans[:38], close(blind_ans)))
print("WITH memory    : %-38s correct=%s" % (informed_ans[:38], close(informed_ans)))

lesson(s) under test:
- Verify unit conversions by checking orders of magnitude; dividing by 100 reduces the value by two decimal places, not one.
- Divide the population density per square kilometre by 100 to convert to people per hectare.



WITHOUT memory : 1314.9                                 correct=False
WITH memory    : 15400000 / 1171 = 13151.15... per km². correct=True


### 9. Failure modes to design against

**The reflection loop.** The actor keeps failing, the reflector keeps producing
lessons, and you burn your budget. *Fix:* a hard `MAX_ATTEMPTS` cap (we have one),
plus a check that reflections are not simply repeating themselves.

**The flattering evaluator.** An LLM asked "is this good?" says yes. *Fix:* ground
the evaluator in execution or a deterministic check, as we did. If you must use an
LLM judge, force it to quote specific evidence and to state a rubric score.

**Vague reflections.** "Be more careful" teaches nothing. *Fix:* make the evaluator's
evidence specific (ours names the exact 100x unit error) and prompt the reflector
for an imperative sentence.

**Cost.** Reflexion is the most expensive pattern in this module: each retry pays for
a full actor run plus a reflection call. Reserve it for tasks where being right
matters more than being cheap.

**Stale reflections.** Lessons from attempt 1 may be irrelevant by attempt 5 and
start misleading the actor. *Fix:* cap the memory length, or have the reflector
rewrite the whole memory instead of appending to it.

In [13]:
print("Repeated-reflection detector (worth running in production):")
seen = set()
for r in out["reflections"]:
    key = r.lower()[:40]
    flag = "DUPLICATE - the actor is stuck" if key in seen else "new"
    print("  %-45s %s" % (r[:45], flag))
    seen.add(key)
if not out["reflections"]:
    print("  (no reflections were needed - the actor passed on attempt 1)")

Repeated-reflection detector (worth running in production):
  Verify unit conversions by checking orders of new
  Divide the population density per square kilo new


### 10. Where Reflexion belongs

Use it when **verification is cheap and objective but generation is hard**:

- Code generation with a test suite - run the tests; failures are the evaluator.
- SQL generation - execute it; a database error message is perfect evidence.
- Structured extraction against a schema - validation errors are the evidence.
- Any task where checking the answer is cheaper than producing it.

Avoid it when you have no grounded evaluator. Reflexion with a self-grading LLM is
mostly a way to spend three times the tokens for the same answer.

### Recap

- Reflexion = actor + **grounded** evaluator + reflector, with lessons carried into
  the retry.
- The memory must **accumulate** (`Annotated[list, operator.add]`), not overwrite.
- Evaluator quality is the whole ballgame; prefer execution over opinion.
- Cap the attempts, watch for repeated reflections, expect a high token bill.

### Next

**[03_plan_and_execute](03_plan_and_execute.ipynb)** - stop deciding one step at a time.